# Incident Response Runbook: Reconnaissance - Active Scanning

**Tactic:** Reconnaissance
**Technique:** Active Scanning (T1595)
**Severity:** MEDIUM

## Overview

This runbook provides step-by-step incident response procedures for Active Scanning activities.

## Incident Response Phases

1. **Detection & Analysis**
2. **Containment**
3. **Eradication**
4. **Recovery**
5. **Post-Incident Activities**


## Phase 1: Detection & Analysis

### Objectives
- Confirm the incident
- Identify the scope
- Assess impact


In [ ]:
import json
import re
from datetime import datetime
import sys
import os

# Add the project root to the Python path
sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), '..', '..')))

from splunk.splunk_data_collector import SplunkDataCollector
from crowdstrike.crowdstrike_response import CrowdStrikeResponse
from iris.iris_integration import IRISIntegration
from misp.misp_integration import MISPIntegration
from shuffle.shuffle_integration import ShuffleIntegration

# Initialize integrations
splunk = SplunkDataCollector()
crowdstrike = CrowdStrikeResponse()
iris = IRISIntegration()
misp = MISPIntegration()
shuffle = ShuffleIntegration()

# Step 1: Detection & Analysis
print("=" * 60)
print("STEP 1: Detection & Analysis")
print("=" * 60)

detection_time = datetime.now().isoformat()

# Query Splunk for active scanning indicators
print(f"\n[QUERY] Searching Splunk for active scanning indicators...")
splunk_query = '''
index=* (sourcetype=WinEventLog:Security EventCode=5156 OR EventCode=5157 OR EventCode=5158)
OR (sourcetype=linux_secure "nmap" OR "masscan" OR "zmap" OR "unicornscan" OR "nikto" OR "dirb" OR "gobuster")
OR (sourcetype=network "TCP SYN" OR "TCP ACK" OR "UDP" OR "ICMP" dest_port!="80" dest_port!="443")
OR (sourcetype=firewall action="blocked" OR action="denied")
OR (sourcetype=ids signature="*scan*" OR signature="*recon*" OR signature="*portscan*")
| eval scanner_ip=coalesce(src_ip, SourceAddress, source_ip)
| eval target_ip=coalesce(dest_ip, DestinationAddress, dest_ip)
| eval scan_type=case(match(_raw, "nmap"), "NMAP", match(_raw, "masscan"), "MASSCAN", match(_raw, "zmap"), "ZMAP", match(_raw, "nikto"), "WEB_SCAN", true(), "UNKNOWN")
| stats count by scanner_ip, target_ip, scan_type, dest_port
| where count > 50
'''

try:
    splunk_results = splunk.search_events(splunk_query, timeframe="-24h")
    print(f"   Found {len(splunk_results)} potential active scanning events")
except Exception as e:
    print(f"   Splunk query failed: {e}")
    splunk_results = []

# Extract affected systems and scanning activities
affected_systems = []
scanning_activities = []
splunk_indicators = []

for event in splunk_results:
    system_info = {
        'scanner_ip': event.get('scanner_ip', 'unknown'),
        'target_ip': event.get('target_ip', 'unknown'),
        'scan_type': event.get('scan_type', 'unknown'),
        'ports_scanned': event.get('dest_port', 'multiple'),
        'scan_count': event.get('count', 0),
        'last_seen': event.get('_time', detection_time)
    }
    affected_systems.append(system_info)

    # Extract indicators
    if event.get('scanner_ip') and event['scanner_ip'] != 'unknown':
        splunk_indicators.append({
            'type': 'ip',
            'value': event.get('scanner_ip'),
            'context': f"Active scanning from {event.get('scanner_ip')} targeting {event.get('target_ip', 'multiple hosts')}"
        })

# Query CrowdStrike for scanning detections
print(f"\n[QUERY] Checking CrowdStrike for scanning detections...")
try:
    cs_detections = crowdstrike.get_detections(
        filter="technique:'Active Scanning'",
        start_time="-24h"
    )
    cs_scanners = []
    for detection in cs_detections:
        scanner_info = {
            'scanner_ip': detection.get('source_ip', ''),
            'hostname': detection.get('hostname', ''),
            'detection_id': detection.get('detection_id', ''),
            'technique': detection.get('technique', ''),
            'severity': detection.get('severity', 0),
            'scan_details': detection.get('scan_details', {})
        }
        cs_scanners.append(scanner_info)

        # Merge with Splunk data
        existing_system = next((s for s in affected_systems if s['scanner_ip'] == scanner_info['scanner_ip']), None)
        if existing_system:
            existing_system.update(scanner_info)
        else:
            affected_systems.append(scanner_info)

    print(f"   Found {len(cs_scanners)} CrowdStrike scanning detections")
except Exception as e:
    print(f"   CrowdStrike query failed: {e}")
    cs_scanners = []

# Enrich with threat intelligence from MISP
print(f"\n[ENRICHMENT] Checking MISP for related threat intelligence...")
misp_results = []
try:
    for indicator in splunk_indicators:
        misp_search = misp.search_iocs(indicator['value'])
        if misp_search:
            misp_results.extend(misp_search)
            print(f"   Found threat intelligence for {indicator['value']}")
except Exception as e:
    print(f"   MISP enrichment failed: {e}")

# Create IRIS case
print(f"\n[CASE] Creating IRIS incident case...")
try:
    incident_data = {
        'title': f'Active Scanning Incident - {len(affected_systems)} scanning sources',
        'description': f'Detected active scanning activities from {len(affected_systems)} sources',
        'severity': 'MEDIUM',
        'tactic': 'Reconnaissance',
        'technique': 'Active Scanning (T1595)',
        'affected_systems': affected_systems,
        'indicators': splunk_indicators,
        'threat_intel': misp_results,
        'detection_time': detection_time
    }
    incident_id = iris.create_case(incident_data)
    print(f"   Created IRIS case: {incident_id}")
except Exception as e:
    print(f"   IRIS case creation failed: {e}")
    incident_id = f"TEMP-{datetime.now().strftime('%Y%m%d-%H%M%S')}"

print(f"\n Detection complete:")
print(f"  - {len(affected_systems)} scanning sources identified")
print(f"  - {len(splunk_indicators)} scanning indicators found")
print(f"  - {len(misp_results)} threat intelligence matches")
print(f"  - IRIS case {incident_id} created")


## Phase 2: Containment

### Objectives
- Stop the attack
- Isolate affected systems
- Prevent spread


In [ ]:
# Step 2: Containment
print("\n" + "=" * 60)
print("STEP 2: Containment")
print("=" * 60)

containment_time = datetime.now().isoformat()
isolated_hosts = []
blocked_ips = []
disabled_scanners = []

# Isolate affected systems
print(f"\n[ACTION] Isolating affected systems...")
for system in affected_systems:
    try:
        if 'device_id' in system:
            isolation_result = crowdstrike.isolate_host(system['device_id'])
            if isolation_result.get('status') == 'isolated':
                isolated_hosts.append(system.get('hostname', system.get('scanner_ip', 'unknown')))
                print(f"   Isolated host: {system.get('hostname', system.get('scanner_ip', 'unknown'))}")
    except Exception as e:
        print(f"   Host isolation failed for {system.get('hostname', system.get('scanner_ip', 'unknown'))}: {e}")

# Block malicious scanning IPs
print(f"\n[ACTION] Blocking malicious scanning IPs...")
unique_ips = set()
for system in affected_systems:
    if system.get('scanner_ip') and system['scanner_ip'] != 'unknown':
        unique_ips.add(system['scanner_ip'])

for ip in unique_ips:
    try:
        block_result = shuffle.block_ip_address(ip)
        if block_result.get('status') == 'blocked':
            blocked_ips.append(ip)
            print(f"   Blocked scanning IP: {ip}")
    except Exception as e:
        print(f"   IP blocking failed for {ip}: {e}")

# Disable scanning tools and processes
print(f"\n[ACTION] Disabling scanning tools and processes...")
for system in affected_systems:
    try:
        if 'device_id' in system:
            scan_disable = crowdstrike.disable_scanning_tools(system['device_id'])
            if scan_disable.get('status') == 'disabled':
                disabled_scanners.append(system.get('hostname', system.get('scanner_ip', 'unknown')))
                print(f"   Disabled scanning tools on {system.get('hostname', system.get('scanner_ip', 'unknown'))}")
    except Exception as e:
        print(f"   Scanning tool disable failed for {system.get('hostname', system.get('scanner_ip', 'unknown'))}: {e}")

# Enable enhanced monitoring
print(f"\n[ACTION] Enabling enhanced monitoring...")
monitoring_rules = []
try:
    # Enable CrowdStrike scanning detection
    cs_monitoring = crowdstrike.enable_enhanced_monitoring('active_scanning')
    if cs_monitoring.get('status') == 'enabled':
        monitoring_rules.append('CrowdStrike scanning monitoring')
        print(f"   Enabled CrowdStrike active scanning monitoring")

    # Enable Splunk scanning correlation rules
    splunk_monitoring = splunk.enable_correlation_rule('scanning_detection')
    if splunk_monitoring.get('status') == 'enabled':
        monitoring_rules.append('Splunk scanning correlation')
        print(f"   Enabled Splunk scanning correlation rules")
except Exception as e:
    print(f"   Enhanced monitoring setup failed: {e}")

# Update IRIS case with containment actions
print(f"\n[ACTION] Updating IRIS case with containment actions...")
try:
    containment_summary = {
        'isolated_hosts': len(isolated_hosts),
        'blocked_ips': len(blocked_ips),
        'disabled_scanners': len(disabled_scanners),
        'monitoring_enabled': len(monitoring_rules),
        'containment_time': containment_time
    }
    iris.update_case(incident_id, 'containment_complete', containment_summary)
    print(f"   Updated IRIS case {incident_id} with containment results")
except Exception as e:
    print(f"   IRIS case update failed: {e}")

print(f"\n Containment complete:")
print(f"  - {len(isolated_hosts)} hosts isolated")
print(f"  - {len(blocked_ips)} scanning IPs blocked")
print(f"  - {len(disabled_scanners)} scanning tools disabled")
print(f"  - {len(monitoring_rules)} enhanced monitoring rules enabled")


## Phase 3: Eradication

### Objectives
- Remove malicious artifacts
- Close vulnerabilities
- Verify threat removal


In [ ]:
# Step 3: Eradication
print("\n" + "=" * 60)
print("STEP 3: Eradication")
print("=" * 60)

removed_scanners = []
patched_systems = []
hardened_network = []

# Remove scanning tools and artifacts
print(f"\n[ACTION] Removing scanning tools and artifacts...")
for system in affected_systems:
    try:
        if 'device_id' in system:
            scanner_removal = crowdstrike.remove_scanning_artifacts(system['device_id'])
            if scanner_removal.get('status') == 'removed':
                removed_scanners.extend(scanner_removal.get('removed_files', []))
                print(f"   Removed {len(scanner_removal.get('removed_files', []))} scanning artifacts from {system.get('hostname', system.get('scanner_ip', 'unknown'))}")
    except Exception as e:
        print(f"   Scanner removal failed for {system.get('hostname', system.get('scanner_ip', 'unknown'))}: {e}")

# Patch systems to prevent scanning
print(f"\n[ACTION] Patching systems to prevent scanning...")
for system in affected_systems:
    try:
        if 'device_id' in system:
            patch_result = crowdstrike.apply_security_patches(system['device_id'])
            if patch_result.get('status') == 'patched':
                patched_systems.append(system.get('hostname', system.get('scanner_ip', 'unknown')))
                print(f"   Applied security patches to {system.get('hostname', system.get('scanner_ip', 'unknown'))}")
    except Exception as e:
        print(f"   Patching failed for {system.get('hostname', system.get('scanner_ip', 'unknown'))}: {e}")

# Harden network perimeter
print(f"\n[ACTION] Hardening network perimeter...")
try:
    network_hardening = shuffle.harden_network_perimeter()
    if network_hardening.get('status') == 'hardened':
        hardened_network.append('perimeter_firewall')
        print(f"   Hardened network perimeter firewall rules")

    ids_hardening = shuffle.enable_intrusion_detection()
    if ids_hardening.get('status') == 'enabled':
        hardened_network.append('intrusion_detection')
        print(f"   Enabled enhanced intrusion detection")
except Exception as e:
    print(f"   Network hardening failed: {e}")

# Clean scanning logs
print(f"\n[ACTION] Cleaning scanning logs...")
cleaned_logs = []
for system in affected_systems:
    try:
        if 'device_id' in system:
            log_cleaning = crowdstrike.clean_scanning_logs(system['device_id'])
            if log_cleaning.get('status') == 'cleaned':
                cleaned_logs.append(system.get('hostname', system.get('scanner_ip', 'unknown')))
                print(f"   Cleaned scanning logs on {system.get('hostname', system.get('scanner_ip', 'unknown'))}")
    except Exception as e:
        print(f"   Log cleaning failed for {system.get('hostname', system.get('scanner_ip', 'unknown'))}: {e}")

# Verify threat removal
print(f"\n[ACTION] Verifying threat removal...")
verification_results = []
for system in affected_systems:
    try:
        if 'device_id' in system:
            verify_result = crowdstrike.verify_scanning_removal(system['device_id'])
            verification_results.append({
                'system': system.get('hostname', system.get('scanner_ip', 'unknown')),
                'status': 'clean' if verify_result.get('scanning_detected', True) == False else 'threats_remaining',
                'remaining_indicators': verify_result.get('remaining_indicators', 0)
            })
            status = "" if verify_result.get('scanning_detected', True) == False else ""
            print(f"  {status} Verification for {system.get('hostname', system.get('scanner_ip', 'unknown'))}: {verify_result.get('remaining_indicators', 0)} scanning indicators remaining")
    except Exception as e:
        print(f"   Verification failed for {system.get('hostname', system.get('scanner_ip', 'unknown'))}: {e}")

print(f"\n Eradication complete:")
print(f"  - {len(removed_scanners)} scanning artifacts removed")
print(f"  - {len(patched_systems)} systems patched")
print(f"  - {len(hardened_network)} network hardening measures applied")
print(f"  - {len(cleaned_logs)} systems had logs cleaned")
print(f"  - {len([v for v in verification_results if v['status'] == 'clean'])} systems verified clean")


## Phase 4: Recovery

### Objectives
- Restore systems
- Validate functionality
- Monitor for issues


In [ ]:
# Step 4: Recovery
print("\n" + "=" * 60)
print("STEP 4: Recovery")
print("=" * 60)

restored_systems = []
validated_functions = []
monitoring_restored = []

# Restore systems from isolation
print(f"\n[ACTION] Restoring systems from isolation...")
for system in affected_systems:
    try:
        if 'device_id' in system and system.get('hostname'):
            restore_result = crowdstrike.restore_network_connectivity(system['device_id'])
            if restore_result.get('status') == 'restored':
                restored_systems.append(system['hostname'])
                print(f"   Restored connectivity for: {system['hostname']}")
    except Exception as e:
        print(f"   System restoration failed for {system.get('hostname', 'unknown')}: {e}")

# Validate system functionality
print(f"\n[ACTION] Validating system functionality...")
functional_tests = []
for system in affected_systems:
    try:
        if 'device_id' in system:
            test_result = crowdstrike.perform_functional_testing(system['device_id'])
            functional_tests.append({
                'system': system.get('hostname', system.get('scanner_ip', 'unknown')),
                'tests_passed': test_result.get('tests_passed', 0),
                'total_tests': test_result.get('total_tests', 0)
            })
            pass_rate = test_result.get('tests_passed', 0) / max(test_result.get('total_tests', 1), 1) * 100
            status = "" if pass_rate >= 95 else ""
            print(f"  {status} Functional tests for {system.get('hostname', system.get('scanner_ip', 'unknown'))}: {test_result.get('tests_passed', 0)}/{test_result.get('total_tests', 0)} passed ({pass_rate:.1f}%)")
    except Exception as e:
        print(f"   Functional testing failed for {system.get('hostname', system.get('scanner_ip', 'unknown'))}: {e}")

# Restore monitoring capabilities
print(f"\n[ACTION] Restoring monitoring capabilities...")
try:
    monitoring_restore = splunk.restore_monitoring()
    if monitoring_restore.get('status') == 'restored':
        monitoring_restored.append('splunk_monitoring')
        print(f"   Restored Splunk monitoring capabilities")

    cs_monitoring_restore = crowdstrike.restore_monitoring()
    if cs_monitoring_restore.get('status') == 'restored':
        monitoring_restored.append('crowdstrike_monitoring')
        print(f"   Restored CrowdStrike monitoring capabilities")
except Exception as e:
    print(f"   Monitoring restoration failed: {e}")

# Monitor for residual issues
print(f"\n[ACTION] Establishing monitoring for residual issues...")
residual_monitoring = []
try:
    # Set up continuous monitoring
    monitoring_setup = splunk.setup_continuous_monitoring('active_scanning')
    if monitoring_setup.get('status') == 'enabled':
        residual_monitoring.append('Splunk continuous monitoring')
        print(f"   Established continuous Splunk monitoring")

    cs_monitoring = crowdstrike.setup_residual_monitoring('reconnaissance')
    if cs_monitoring.get('status') == 'enabled':
        residual_monitoring.append('CrowdStrike residual monitoring')
        print(f"   Established CrowdStrike residual monitoring")
except Exception as e:
    print(f"   Residual monitoring setup failed: {e}")

print(f"\n Recovery complete:")
print(f"  - {len(restored_systems)} systems restored from isolation")
print(f"  - {len([f for f in functional_tests if f['tests_passed'] == f['total_tests']])} systems passed all functional tests")
print(f"  - {len(monitoring_restored)} monitoring systems restored")
print(f"  - {len(residual_monitoring)} residual monitoring systems established")


## Phase 5: Post-Incident Activities

### Objectives
- Document findings
- Implement improvements
- Share lessons learned


In [ ]:
# Step 5: Post-Incident
print("\n" + "=" * 60)
print("STEP 5: Post-Incident")
print("=" * 60)

# Update IRIS case with eradication results
print(f"\n[ACTION] Updating IRIS case with eradication results...")
try:
    eradication_summary = {
        'removed_scanners': len(removed_scanners),
        'patched_systems': len(patched_systems),
        'hardened_network': len(hardened_network),
        'cleaned_logs': len(cleaned_logs),
        'verification_results': verification_results
    }
    iris.update_case(incident_id, 'eradication_complete', eradication_summary)
    print(f"   Updated IRIS case {incident_id} with eradication results")
except Exception as e:
    print(f"   IRIS case update failed: {e}")

# Update IRIS case with recovery results
print(f"\n[ACTION] Updating IRIS case with recovery results...")
try:
    recovery_summary = {
        'restored_systems': len(restored_systems),
        'functional_tests': functional_tests,
        'monitoring_restored': len(monitoring_restored),
        'residual_monitoring': len(residual_monitoring)
    }
    iris.update_case(incident_id, 'recovery_complete', recovery_summary)
    print(f"   Updated IRIS case {incident_id} with recovery results")
except Exception as e:
    print(f"   IRIS case update failed: {e}")

# Generate incident report
print(f"\n[ACTION] Generating incident report...")
try:
    incident_report = {
        'incident_id': incident_id,
        'technique': 'Reconnaissance: Active Scanning',
        'detection_time': detection_time,
        'affected_systems': len(affected_systems),
        'timeline': {
            'detection': detection_time,
            'containment': containment_time,
            'eradication': datetime.now().isoformat(),
            'recovery': datetime.now().isoformat()
        },
        'actions_taken': {
            'detection': f"Found {len(affected_systems)} scanning sources",
            'containment': f"Isolated {len(isolated_hosts)} hosts, blocked {len(blocked_ips)} IPs",
            'eradication': f"Removed {len(removed_scanners)} scanning artifacts, patched {len(patched_systems)} systems",
            'recovery': f"Restored {len(restored_systems)} systems, validated {len([f for f in functional_tests if f['tests_passed'] == f['total_tests']])} functions"
        },
        'indicators': splunk_indicators,
        'threat_intel': misp_results,
        'lessons_learned': [
            "Implement comprehensive network scanning detection",
            "Deploy network intrusion detection systems",
            "Regular vulnerability scanning of own infrastructure",
            "Implement network segmentation to limit reconnaissance",
            "Enable detailed logging and monitoring of network traffic"
        ]
    }
    iris.generate_report(incident_id, incident_report)
    print(f"   Generated comprehensive incident report for case {incident_id}")
except Exception as e:
    print(f"   Report generation failed: {e}")

# Share IOCs with MISP
print(f"\n[ACTION] Sharing IOCs with MISP...")
try:
    misp_iocs = []
    for indicator in splunk_indicators:
        if indicator.get('type') in ['ip']:
            misp_iocs.append({
                'type': indicator['type'],
                'value': indicator['value'],
                'tags': ['active-scanning', 'reconnaissance', 'incident-' + str(incident_id)]
            })

    if misp_iocs:
        misp.share_iocs(misp_iocs, f"Active Scanning Incident {incident_id}")
        print(f"   Shared {len(misp_iocs)} IOCs with MISP community")
    else:
        print(f"   No new IOCs to share with MISP")
except Exception as e:
    print(f"   MISP IOC sharing failed: {e}")

# Close IRIS case
print(f"\n[ACTION] Closing IRIS case...")
try:
    iris.close_case(incident_id, 'resolved')
    print(f"   Closed IRIS case {incident_id} as resolved")
except Exception as e:
    print(f"   IRIS case closure failed: {e}")

# Final status summary
print(f"\n Post-incident activities complete:")
print(f"  - IRIS case {incident_id} updated and closed")
print(f"  - Incident report generated")
print(f"  - {len(misp_iocs) if 'misp_iocs' in locals() else 0} IOCs shared with threat intelligence community")
print(f"  - All 5 IR phases completed successfully")

# Calculate incident metrics
incident_duration = (datetime.now() - datetime.fromisoformat(detection_time)).total_seconds() / 3600
print(f"\n📊 Incident Metrics:")
print(f"  - Duration: {incident_duration:.2f} hours")
print(f"  - Systems affected: {len(affected_systems)}")
print(f"  - Containment time: {'< 1 hour' if incident_duration < 1 else f'{incident_duration:.1f} hours'}")
print(f"  - Eradication success: {len([v for v in verification_results if v['status'] == 'clean'])}/{len(affected_systems)} systems clean")
print(f"  - Recovery success: {len([f for f in functional_tests if f['tests_passed'] == f['total_tests']])}/{len(affected_systems)} systems fully functional")


## Summary

This incident response runbook has guided you through the complete lifecycle of responding to this incident.

### Key Takeaways
- Rapid detection and response are critical
- Containment prevents further damage
- Thorough eradication prevents reinfection
- Continuous monitoring ensures recovery

### Next Steps
- Review and approve the incident report
- Implement recommended preventive measures
- Schedule follow-up review
- Share lessons learned with the team
